# DSHI spectrum (FR vs SIL)

In [ ]:
import colorsys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


def hsl_to_rgb_normalized(h, s, l):
    return colorsys.hls_to_rgb(h / 360, l / 100, s / 100)


BLUE_RGB = hsl_to_rgb_normalized(206, 73, 48)
RED_RGB = hsl_to_rgb_normalized(9, 98, 63)

plt.rcParams.update(
    {
        "font.size": 24,
        "axes.titlesize": 24,
        "axes.labelsize": 24,
        "xtick.labelsize": 24,
        "ytick.labelsize": 24,
        "legend.fontsize": 24,
    }
)

In [ ]:
def _resolve_data_dir() -> Path:
    base = Path("..") / "data"
    sub = base / "Характеристики излучения"
    if sub.is_dir():
        return sub
    return base


def load_spectrum_csv(path: Path) -> pd.DataFrame:
    # 18 строк метаданных, затем таблица.
    # Нужны последние 2 числовые колонки: power_linear и normalized_power.
    df = pd.read_csv(
        path,
        header=None,
        skiprows=18,
        usecols=[0, 3, 4],
        names=["frequency_hz", "power_linear", "normalized_power"],
    )

    for col in ["frequency_hz", "power_linear", "normalized_power"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna().sort_values("frequency_hz").reset_index(drop=True)
    if df.empty:
        raise ValueError(f"{path} не содержит валидных данных")

    # Центрируем ось частоты относительно максимума нормированной мощности.
    center_hz = float(df.loc[df["normalized_power"].idxmax(), "frequency_hz"])
    df["detuning_hz"] = df["frequency_hz"] - center_hz
    return df, center_hz


def load_fit_csv(path: Path, center_hz: float) -> pd.DataFrame:
    # Формат fit-файлов: первая колонка — X (frequency), вторая — Voigt Fit.
    # Остальные колонки игнорируем.
    df = pd.read_csv(path, header=0, usecols=[0, 1])
    df.columns = ["frequency_hz", "fit_normalized_power"]

    for col in ["frequency_hz", "fit_normalized_power"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna().sort_values("frequency_hz").reset_index(drop=True)
    if df.empty:
        raise ValueError(f"{path} не содержит валидных данных fit")

    df["detuning_hz"] = df["frequency_hz"] - center_hz
    return df


DATA_DIR = _resolve_data_dir()
fr, fr_center_hz = load_spectrum_csv(DATA_DIR / "Spectrum_FR.csv")
sil, sil_center_hz = load_spectrum_csv(DATA_DIR / "Spectrum_SIL.csv")

fr_fit = load_fit_csv(DATA_DIR / "Spectrum_FR_fit.csv", center_hz=fr_center_hz)
sil_fit = load_fit_csv(DATA_DIR / "Spectrum_SIL_fit.csv", center_hz=sil_center_hz)

print("FR points:", len(fr), "fit points:", len(fr_fit), "detuning range [MHz]:", fr["detuning_hz"].min() / 1e6, "..", fr["detuning_hz"].max() / 1e6)
print("SIL points:", len(sil), "fit points:", len(sil_fit), "detuning range [kHz]:", sil["detuning_hz"].min() / 1e3, "..", sil["detuning_hz"].max() / 1e3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

x_fr_mhz = fr["detuning_hz"].to_numpy() / 1e6
x_fr_fit_mhz = fr_fit["detuning_hz"].to_numpy() / 1e6

ax = axes[0]
ax.plot(x_fr_mhz, fr["normalized_power"], color=BLUE_RGB, linewidth=1.0, label="Free-run")
ax.plot(
    x_fr_fit_mhz,
    fr_fit["fit_normalized_power"],
    color=RED_RGB,
    linestyle="--",
    linewidth=1.0,
    label="Voigt fit",
)
ax.set_yscale("log")
ax.set_xlim(-50, 50)
ax.set_ylim(1e-5, 2)
ax.set_xlabel("Frequency, MHz")
ax.set_ylabel("Normalized power")
ax.grid(False)
ax.legend(loc="upper right", frameon=True)

x_sil_khz = sil["detuning_hz"].to_numpy() / 1e3
x_sil_fit_khz = sil_fit["detuning_hz"].to_numpy() / 1e3

ax = axes[1]
ax.plot(x_sil_khz, sil["normalized_power"], color=BLUE_RGB, linewidth=1.0, label="SIL spectrum")
ax.plot(
    x_sil_fit_khz,
    sil_fit["fit_normalized_power"],
    color=RED_RGB,
    linestyle="--",
    linewidth=1.0,
    label="Voigt fit",
)
ax.set_yscale("log")
ax.set_xlim(-100, 100)
ax.set_ylim(1e-7, 2)
ax.set_xlabel("Frequency, kHz")
ax.set_ylabel("Normalized power")
ax.grid(False)
ax.legend(loc="upper right", frameon=True)

plt.tight_layout()
plt.show()